# Discovery, Validation, And Artifact Inspection

This notebook runs discovery and validation, then loads the cluster summary and validation outputs so you can inspect motifs without reading raw JSON by hand.

Default behavior is intentionally simple: use the full canonical dataset unless you explicitly turn subset selection back on.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import json
import pandas as pd

drive.mount('/content/drive')
repo_root = Path('/content/predictive-circuit-coding')
drive_root = Path('/content/drive/MyDrive/pcc_runs')
drive_data_root = drive_root / 'data' / 'allen_visual_behavior_neuropixels'
drive_artifacts_root = drive_root / 'artifacts'
drive_artifacts_root.mkdir(parents=True, exist_ok=True)
print('Drive mounted.')
print('Repo root:', repo_root)
print('Drive dataset root:', drive_data_root)
print('Drive artifacts root:', drive_artifacts_root)

In [ ]:
if not repo_root.exists():
    !git clone https://github.com/jacobposchl/predictive-circuit-coding.git {repo_root}
%cd {repo_root}
!pip install -e .[notebook]

In [ ]:
repo_dataset_root = repo_root / 'data' / 'allen_visual_behavior_neuropixels'
repo_artifacts_root = repo_root / 'artifacts'

assert drive_data_root.exists(), f'Missing Drive dataset bundle: {drive_data_root}'

repo_dataset_root.parent.mkdir(parents=True, exist_ok=True)
if repo_dataset_root.exists() or repo_dataset_root.is_symlink():
    if repo_dataset_root.is_symlink():
        repo_dataset_root.unlink()
    else:
        shutil.rmtree(repo_dataset_root)
repo_dataset_root.symlink_to(drive_data_root, target_is_directory=True)

if repo_artifacts_root.exists() or repo_artifacts_root.is_symlink():
    if repo_artifacts_root.is_symlink():
        repo_artifacts_root.unlink()
    else:
        shutil.rmtree(repo_artifacts_root)
repo_artifacts_root.symlink_to(drive_artifacts_root, target_is_directory=True)

print('Linked dataset into repo:', repo_dataset_root)
print('Linked artifacts into repo:', repo_artifacts_root)

In [ ]:
import subprocess
import yaml
import ipywidgets as widgets
from predictive_circuit_coding.utils import NotebookStageReporter, verify_paths_exist

reporter = NotebookStageReporter(name='colab-discover', expected_duration='1-5 minutes for debug runs, longer for larger discovery passes')
reporter.banner('Discovery And Validation', subtitle='Frozen-token discovery, validation controls, and report inspection')

base_experiment_config = repo_root / 'configs' / 'pcc' / 'predictive_circuit_coding_base.yaml'
data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
runtime_experiment_config = repo_root / 'artifacts' / 'colab_runtime_experiment.yaml'

USE_FULL_DATASET = True

selection_disabled = {
    'output_name': 'full_dataset_colab',
    'session_ids': [],
    'subject_ids': [],
    'exclude_session_ids': [],
    'exclude_subject_ids': [],
    'session_ids_file': None,
    'subject_ids_file': None,
    'exclude_session_ids_file': None,
    'exclude_subject_ids_file': None,
    'experience_levels': [],
    'session_types': [],
    'image_sets': [],
    'session_numbers': [],
    'project_codes': [],
    'brain_regions_any': [],
    'min_n_units': None,
    'max_n_units': None,
    'min_trial_count': None,
    'max_trial_count': None,
    'min_duration_s': None,
    'max_duration_s': None,
    'split_seed': None,
    'split_primary_axis': None,
    'train_fraction': None,
    'valid_fraction': None,
    'discovery_fraction': None,
    'test_fraction': None,
}

experiment_payload = yaml.safe_load(base_experiment_config.read_text(encoding='utf-8'))
if USE_FULL_DATASET:
    experiment_payload['dataset_selection'] = selection_disabled

runtime_experiment_config.write_text(yaml.safe_dump(experiment_payload, sort_keys=False), encoding='utf-8')
experiment_config = runtime_experiment_config

dataset_selection_active = any(
    value not in (None, [], '', {})
    for key, value in experiment_payload.get('dataset_selection', {}).items()
    if key != 'output_name'
)

checkpoint_dir = repo_root / 'artifacts' / 'checkpoints'
checkpoint_options = sorted(str(path) for path in checkpoint_dir.glob('*_best.pt'))
checkpoint_widget = widgets.Dropdown(options=checkpoint_options, description='Checkpoint:')
display(checkpoint_widget)
assert checkpoint_options, 'No best checkpoints found. Run the training notebook first.'
print('Experiment config:', experiment_config)
print('Data config:', data_config)
print('Runtime selection active:', dataset_selection_active)

In [ ]:
selected_checkpoint = checkpoint_widget.value
paths_ok = verify_paths_exist({
    'experiment_config': experiment_config,
    'data_config': data_config,
    'checkpoint': selected_checkpoint,
    'drive_dataset_root': drive_data_root,
})
print(paths_ok)
assert all(paths_ok.values()), 'Missing discovery/validation inputs.'

if dataset_selection_active:
    reporter.begin('runtime selection', next_artifact='selected split manifest')
    subprocess.run([
        'pcc-prepare-data',
        'materialize-runtime-selection',
        '--config', str(experiment_config),
        '--data-config', str(data_config),
    ], check=True)
    reporter.finish('runtime selection')
else:
    print('Using the full canonical dataset. No runtime subset will be materialized.')

In [ ]:
reporter.begin('discovery', next_artifact='discovery artifact + cluster summary json/csv')
%cd {repo_root}
subprocess.run([
    'pcc-discover',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
    '--checkpoint', str(selected_checkpoint),
    '--split', 'discovery',
], check=True)
reporter.finish('discovery')

In [ ]:
discovery_artifact = Path(selected_checkpoint).with_name(f"{Path(selected_checkpoint).stem}_discovery_discovery.json")
cluster_summary_json = discovery_artifact.with_name(f"{discovery_artifact.stem}_cluster_summary.json")
cluster_summary_csv = discovery_artifact.with_name(f"{discovery_artifact.stem}_cluster_summary.csv")
reporter.begin('validation', next_artifact='validation summary json/csv')
%cd {repo_root}
subprocess.run([
    'pcc-validate',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
    '--checkpoint', str(selected_checkpoint),
    '--discovery-artifact', str(discovery_artifact),
], check=True)
reporter.finish('validation')

In [ ]:
validation_json = discovery_artifact.with_name(f"{discovery_artifact.stem}_validation.json")
validation_csv = discovery_artifact.with_name(f"{discovery_artifact.stem}_validation.csv")

with open(discovery_artifact, 'r', encoding='utf-8') as handle:
    discovery_payload = json.load(handle)
with open(cluster_summary_json, 'r', encoding='utf-8') as handle:
    cluster_summary_payload = json.load(handle)
with open(validation_json, 'r', encoding='utf-8') as handle:
    validation_payload = json.load(handle)

display(pd.read_csv(cluster_summary_csv))
display(pd.DataFrame(discovery_payload['candidates']).head())
display(pd.read_csv(validation_csv))
print('Cluster count:', cluster_summary_payload['cluster_count'])
print('Recurrence summary:', validation_payload['recurrence_summary'])